In [ ]:
print

In [16]:
import os
import sys
import logging
import numpy as np
import pandas as pd
from sqlalchemy import create_engine
import xgboost as xgb
from sklearn.metrics import roc_auc_score

# Add the parent directory to sys.path to find utils
sys.path.insert(0,'YOUR-PATH/actionable-hypotension')

#from utils.db_interface import get_engine


### Dataset caching for faster multiple training

In [17]:
# Global Cache
DF_FEATURES_CACHE = None
SPLITS_CACHE = None
FEATURE_COLS_CACHE = None
DMATRICES_CACHE = None
DATABASE_URI = "postgresql+psycopg2://USER@localhost:5432/eicu"
engine = create_engine(DATABASE_URI, future=True)

### Config

In [ ]:
MODELS_DIR = "YOUR-PATH/actionable-hypotension/models_eicu/uncalibrated"
DASHBOARDS_DIR = "optuna_dashboards"
LOG_LEVEL = logging.INFO

In [19]:

# ----------------------
# Set up logging
# ----------------------
logging.basicConfig(
    level=LOG_LEVEL,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)

In [20]:

def load_data(table_name: str) -> pd.DataFrame:
    """
    Load data from the specified SQL table.
    """
    
    query = f"SELECT * FROM public.{table_name}"
    df = pd.read_sql(query, engine)
    logging.info(f"Loaded {len(df)} rows, {df.shape[1]} columns")
    return df


def preprocess_data(
    df: pd.DataFrame
) -> pd.DataFrame:
    """
    Drop metadata/time columns and encode the label.
    
    """
    drop_cols = [
        "patienthealthsystemstayid", "patientunitstayid",
        "context_start_offset_min", "context_end_offset_min",
        "target_start_offset_min", "target_end_offset"
    ]
    logging.info("Dropping metadata/time/JSON columns")
    df_features = df.drop(columns=[c for c in drop_cols if c in df.columns])


    # Label as int
    df_features["label"] = df_features["positive_event"].astype(int)

    return df_features


def split_data(df_features: pd.DataFrame) -> dict:
    """
    Split the DataFrame into train, validation, and test sets.
    """
    logging.info("Splitting data into train/val/test")
    return {
        "train": df_features[df_features["split"] == "train"].sample(frac=1, random_state=42).reset_index(drop=True),
        "val":   df_features[df_features["split"] == "val"],
        "test":  df_features[df_features["split"] == "test"],
    }


def get_feature_cols(df_features: pd.DataFrame) -> list:
    """
    Identify feature columns (exclude label and split markers).
    """
    excluded = {"positive_event", "positive_sample", "split", "label"}
    feature_cols = [c for c in df_features.columns if c not in excluded]
    
    # Log the remaining feature columns
    logging.info(f"Using {len(feature_cols)} feature columns: {feature_cols}")
    
    return feature_cols


def create_dmatrices(splits: dict, feature_cols: list) -> dict:
    """
    Create XGBoost DMatrix objects for train, val, and test.
    """
    dmatrices = {}
    for name, subset in splits.items():
        X = subset[feature_cols]
        y = subset["label"]
        logging.info(f"Creating DMatrix for {name} ({len(subset)} rows)")
        dmatrices[name] = xgb.DMatrix(X, label=y, missing=np.nan)
    return dmatrices


def train_model(dtrain, dval, params: dict) -> xgb.Booster:
    """
    Train an XGBoost model with early stopping on the validation set.
    """
    watchlist = [(dtrain, "train"), (dval, "val")]
    logging.info("Starting training")
    bst = xgb.train(
        params,
        dtrain,
        num_boost_round=1000,
        early_stopping_rounds=50,
        evals=watchlist,
        verbose_eval=10,
    )
    return bst


def evaluate_model(bst: xgb.Booster, dtest, y_test: pd.Series) -> float:
    """
    Evaluate the trained model on the test set and log AUC.
    """
    logging.info("Evaluating on test set")
    preds = bst.predict(dtest)
    auc = roc_auc_score(y_test, preds)
    logging.info(f"Test AUC: {auc:.4f}")
    return auc


def save_model(bst: xgb.Booster, model_name: str) -> str:
    """
    Save the model to the models directory using modern JSON format.
    """
    os.makedirs(MODELS_DIR, exist_ok=True)
    
    model_path = os.path.join(MODELS_DIR, f"xgb_{model_name}.json")
    logging.info(f"Saving model to {model_path}")
    
    bst.save_model(model_path)
    return model_path

In [21]:
def prepare_training_data_cached(table_name: str):
    global DF_FEATURES_CACHE, SPLITS_CACHE, FEATURE_COLS_CACHE, DMATRICES_CACHE

    if all(v is not None for v in [DF_FEATURES_CACHE, SPLITS_CACHE, FEATURE_COLS_CACHE, DMATRICES_CACHE]):
        logging.info("Using cached training data.")
        return DF_FEATURES_CACHE, SPLITS_CACHE, FEATURE_COLS_CACHE, DMATRICES_CACHE

    logging.info(f"Loading training data from public.{table_name}")
    
    df = load_data(table_name)

    # Vorverarbeitung
    df_features = preprocess_data(
        df
    )

    splits = split_data(df_features)
    feature_cols = get_feature_cols(df_features)
    dmatrices = create_dmatrices(splits, feature_cols)

    logging.info("✅ Training data prepared.")
    return df_features, splits, feature_cols, dmatrices

### Hyperparameter Optimization on mixed dateset

In [ ]:
def run_optuna_tuning(table_name: str, model_name: str, dashboard_name: str="xgboost_optimization", n_trials: int = 50, drop_treatment_given=False, drop_only_2_values=False):
    """
    Run Optuna optimization for the given table.
    """
    logging.info(f"Starting Optuna tuning on table: {table_name}")

    os.makedirs(DASHBOARDS_DIR, exist_ok=True)

    # Load and prepare data (with caching)
    df_features, splits, feature_cols, dmatrices = prepare_training_data_cached(table_name, drop_treatment_given=drop_treatment_given, drop_only_2_values=drop_only_2_values)

    dtrain = dmatrices["train"]
    dval   = dmatrices["val"]
    y_val  = splits["val"]["label"]

    # Compute scale_pos_weight for class imbalance
    y_train = splits["train"]["label"]
    scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

    def objective(trial):
        params = {
            "nthread": 16,
            "objective": "binary:logistic",
            "eval_metric": "auc",
            "verbosity": 0,
            "tree_method": "hist",
            "max_bin": 512,
            "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.3, log=True),
            "max_depth": trial.suggest_int("max_depth", 3, 15),
            "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),
            "gamma": trial.suggest_float("gamma", 0, 10),
            "subsample": trial.suggest_float("subsample", 0.3, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.3, 1.0),
            "colsample_bylevel": trial.suggest_float("colsample_bylevel", 0.3, 1.0),
            "colsample_bynode": trial.suggest_float("colsample_bynode", 0.3, 1.0),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
            "max_delta_step": trial.suggest_int("max_delta_step", 0, 10),
            "n_estimators": trial.suggest_int("n_estimators", 100, 1000)
        }

        model = xgb.train(
            params,
            dtrain,
            num_boost_round=500,
            early_stopping_rounds=30,
            evals=[(dval, "val")],
            verbose_eval=False
        )

        y_pred_val = model.predict(dval)
        score = roc_auc_score(y_val, y_pred_val)
        return score

    study_name = f"{model_name}"
    storage_path = f"sqlite:///{DASHBOARDS_DIR}/{dashboard_name}.db"

    study = optuna.create_study(
        study_name=study_name,
        direction="maximize",
        sampler=TPESampler(seed=42),
        storage=storage_path,
        load_if_exists=True
    )
    study.optimize(objective, n_trials=n_trials)

    logging.info(f"Best score: {study.best_value:.5f}")
    logging.info(f"Best params: {study.best_params}")

    # Retrain best model on full training set
    best_params = study.best_params
    best_params.update({
        "objective": "binary:logistic",
        "eval_metric": "auc",
        "tree_method": "hist",
        "max_bin": 512,
        "verbosity": 1
    })

    bst = xgb.train(
        best_params,
        dtrain,
        num_boost_round=500,
        early_stopping_rounds=30,
        evals=[(dval, "val")],
        verbose_eval=10
    )

    # Save model
    os.makedirs(MODELS_DIR, exist_ok=True)
    model_path = os.path.join(MODELS_DIR, f"xgb_{model_name}.model")
    save_model(bst, model_name)
    logging.info(f"Saved best model to: {model_path}")

    return bst, study


In [ ]:
bst, study = run_optuna_tuning("merged_mix_features", model_name="mix", n_trials=200, drop_treatment_given=True, drop_only_2_values=True)

### Train mix, invasive and non-invasive models with optimized hyperparameters

In [12]:
def train_model_for_table(table_name: str, model_name: str, override_params: dict = None) -> xgb.Booster:
    df_features, splits, feature_cols, dmatrices = prepare_training_data_cached(table_name)
    
    y_train = splits["train"]["label"]
    neg = (y_train == 0).sum()
    pos = (y_train == 1).sum()
    scale_pos_weight = neg / pos
    logging.info(f"Calculated scale_pos_weight: {scale_pos_weight:.2f}")

    default_params = {
        "objective":     "binary:logistic",
        "eval_metric":   "auc",
        "tree_method":   "hist",
        "learning_rate": 0.01,
        "max_depth":     10,
        "verbosity":     1,
        "scale_pos_weight": scale_pos_weight
    }

    params = override_params or default_params

    bst = train_model(dmatrices["train"], dmatrices["val"], params)
    evaluate_model(bst, dmatrices["test"], splits["test"]["label"])
    save_model(bst, model_name)

    return bst

In [ ]:
# Load study from Optuna database
dashboard_name = "xgboost_optimization"
storage = f"sqlite:///{DASHBOARDS_DIR}/{dashboard_name}.db"
study_name = "mix"

study = optuna.load_study(study_name=study_name, storage=storage)
best_trial = study.best_trial

# Extract best hyperparameters and add / override fixed values
params = best_trial.params.copy()
params.update({
    "objective": "binary:logistic",
    "eval_metric": "auc",
    "tree_method": "hist",
    "nthread": 16,
    "max_bin": 512,
    "verbosity": 1,
    "scale_pos_weight": 1
})

params

In [ ]:
# Round all float hyperparameters to 4 decimals for reproducibility & readability
params = {
    k: round(v, 4) if isinstance(v, float) else v
    for k, v in params.items()
}

params

In [ ]:
print(os.getenv("DATABASE_URI"))

postgresql+psycopg2://USER@localhost/mimic


In [22]:

train_model_for_table(table_name="merged_mix_features", model_name="mix")

2026-02-16 12:13:18 [INFO] Loading training data from public.merged_mix_features
2026-02-16 12:18:29 [INFO] Loaded 28941203 rows, 45 columns
2026-02-16 12:18:29 [INFO] Dropping metadata/time/JSON columns
2026-02-16 12:18:30 [INFO] Splitting data into train/val/test
2026-02-16 12:18:47 [INFO] Using 36 feature columns: ['mean', 'median', 'min', 'max', 'std', 'iqr', 'first', 'last', 'rate_change', 'slope', 'weighted_mean', 'gender_bin', 'ethnicity_bin', 'age_bin', 'admissionheight_bin', 'admissionweight_bin', 'bmi_bin', 'obesity', 'hypertension', 'diabetes', 'kidney_disease', 'lung_disease', 'heart_disease', 'drug_abuse', 'depression', 'sedatives_given', 'blood_products_transfusions_given', 'antibiotics_given', 'anticoagulants_antiplatelets_given', 'neuromuscular_blockers_given', 'analgesics_given', 'crystalloids_given', 'electrolytes_given', 'gi_protection_given', 'parenteral_nutrition_given', 'antiarrhythmics_given']
2026-02-16 12:18:47 [INFO] Creating DMatrix for train (20185817 rows)


[0]	train-auc:0.76467	val-auc:0.69493
[10]	train-auc:0.77455	val-auc:0.69957
[20]	train-auc:0.78002	val-auc:0.70298
[30]	train-auc:0.78780	val-auc:0.70939
[40]	train-auc:0.79128	val-auc:0.71208
[50]	train-auc:0.79352	val-auc:0.71421
[60]	train-auc:0.79578	val-auc:0.71568
[70]	train-auc:0.79804	val-auc:0.71648
[80]	train-auc:0.79997	val-auc:0.71729
[90]	train-auc:0.80203	val-auc:0.71764
[100]	train-auc:0.80390	val-auc:0.71807
[110]	train-auc:0.80547	val-auc:0.71839
[120]	train-auc:0.80700	val-auc:0.71889
[130]	train-auc:0.80882	val-auc:0.71953
[140]	train-auc:0.81073	val-auc:0.72001
[150]	train-auc:0.81241	val-auc:0.72056
[160]	train-auc:0.81410	val-auc:0.72093
[170]	train-auc:0.81574	val-auc:0.72144
[180]	train-auc:0.81772	val-auc:0.72183
[190]	train-auc:0.81991	val-auc:0.72215
[200]	train-auc:0.82133	val-auc:0.72224
[210]	train-auc:0.82304	val-auc:0.72265
[220]	train-auc:0.82473	val-auc:0.72281
[230]	train-auc:0.82658	val-auc:0.72324
[240]	train-auc:0.82865	val-auc:0.72359
[250]	train

2026-02-16 12:36:18 [INFO] Evaluating on test set
2026-02-16 12:36:20 [INFO] Test AUC: 0.7408
2026-02-16 12:36:20 [INFO] Saving model to YOUR-PATH/actionable-hypotension/models/uncalibrated/xgb_mix.json
